# 🏎️ F1 Weekend Overview
## `[RACE NAME]` · `[YEAR]` Formula 1 World Championship

Covers **Qualifying → Sprint (optional) → Race** in one notebook.

**Setup:**
1. Copy this folder to `2026/R{round}_{location}/`
2. Fill in the config cell below
3. Kernel → Restart & Run All
4. Charts land in `assets/` — ready for LinkedIn

---

In [ ]:
# ============================================================
# CONFIGURE — fill these in, leave everything else as-is
# ============================================================
YEAR       = 2026
GRAND_PRIX = 'Barcelona'   # city or GP name recognised by FastF1
HAS_SPRINT = False         # set True for Sprint weekends

# Drivers to highlight in telemetry / pace comparison
# Leave None to auto-pick top 2 race finishers
DRIVER_1 = None
DRIVER_2 = None
# ============================================================

In [ ]:
import sys, os
from pathlib import Path

# Find repo root regardless of where this notebook lives
_here = Path(os.getcwd()).resolve()
_root = _here
for _ in range(5):
    if (_root / 'utils' / 'f1_helpers.py').exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))

from utils.f1_helpers import (
    setup, ensure_assets_dir,
    get_race_results, plot_race_results,
    plot_lap_times, plot_tyre_strategy, plot_telemetry,
    get_qualifying_times, plot_qualifying_times, plot_sector_times,
    plot_positions_gained, plot_pace_comparison,
)
import fastf1

setup('content/f1_cache')
ASSETS = ensure_assets_dir('assets')
print('✅ Libraries loaded')

In [ ]:
# Load all sessions up-front so cache hits are batched
print('Loading Qualifying...')
quali = fastf1.get_session(YEAR, GRAND_PRIX, 'Q')
quali.load(telemetry=False, weather=False)

if HAS_SPRINT:
    print('Loading Sprint Shootout...')
    sprint_quali = fastf1.get_session(YEAR, GRAND_PRIX, 'SQ')
    sprint_quali.load(telemetry=False, weather=False)

    print('Loading Sprint Race...')
    sprint = fastf1.get_session(YEAR, GRAND_PRIX, 'S')
    sprint.load(telemetry=True, weather=False)

print('Loading Race...')
race = fastf1.get_session(YEAR, GRAND_PRIX, 'R')
race.load(telemetry=True, weather=False)

print(f'\n✅ All sessions loaded — {race.event["EventName"]} {YEAR}')

---
## 1 · Qualifying

In [ ]:
quali_df = get_qualifying_times(quali)
quali_df.head(15)

In [ ]:
fig_q1 = plot_qualifying_times(quali, top_n=15, assets_path=ASSETS)
fig_q1.show()

In [ ]:
fig_q2 = plot_sector_times(quali, top_n=10, assets_path=ASSETS)
fig_q2.show()

---
## 2 · Sprint Weekend *(skip if `HAS_SPRINT = False`)*

In [ ]:
if HAS_SPRINT:
    print('=== Sprint Shootout ===')
    fig_sq = plot_qualifying_times(sprint_quali, top_n=15, assets_path=ASSETS)
    # rename file so it doesn't overwrite Q chart
    import os, shutil
    shutil.move(f'{ASSETS}/Q1_qualifying_times.png', f'{ASSETS}/SQ1_shootout_times.png')
    fig_sq.show()
else:
    print('No Sprint this weekend — skipping.')

In [ ]:
if HAS_SPRINT:
    print('=== Sprint Race ===')
    fig_sr1 = plot_race_results(sprint, assets_path=ASSETS)
    shutil.move(f'{ASSETS}/01_race_results.png', f'{ASSETS}/S1_sprint_results.png')
    fig_sr1.show()

    fig_sr2 = plot_lap_times(sprint, assets_path=ASSETS)
    shutil.move(f'{ASSETS}/02_lap_time_evolution.png', f'{ASSETS}/S2_sprint_lap_times.png')
    fig_sr2.show()
else:
    print('No Sprint this weekend — skipping.')

---
## 3 · Race

In [ ]:
race_results = get_race_results(race)
race_results.head(10)

In [ ]:
fig_r1 = plot_race_results(race, ASSETS)
fig_r1.show()

In [ ]:
fig_r2 = plot_lap_times(race, assets_path=ASSETS)
fig_r2.show()

In [ ]:
fig_r3 = plot_tyre_strategy(race, ASSETS)
fig_r3.show()

In [ ]:
if DRIVER_1 is None or DRIVER_2 is None:
    top2 = race_results.head(2)
    DRIVER_1 = race.results[race.results['FullName'] == top2.iloc[0]['FullName']]['Abbreviation'].values[0]
    DRIVER_2 = race.results[race.results['FullName'] == top2.iloc[1]['FullName']]['Abbreviation'].values[0]
print(f'Telemetry comparison: {DRIVER_1} vs {DRIVER_2}')

In [ ]:
fig_r4 = plot_telemetry(race, DRIVER_1, DRIVER_2, ASSETS)
fig_r4.show()

---
## 4 · Weekend Summary

In [ ]:
# Grid position (from qualifying) vs final race position
fig_w1 = plot_positions_gained(race, ASSETS)
fig_w1.show()

In [ ]:
# Median pace per session for the two focus drivers
sessions_map = {'Quali': quali, 'Race': race}
if HAS_SPRINT:
    sessions_map = {'Quali': quali, 'Sprint': sprint, 'Race': race}

fig_w2 = plot_pace_comparison(sessions_map, drivers=[DRIVER_1, DRIVER_2], assets_path=ASSETS)
fig_w2.show()

---
## 5 · LinkedIn Post Draft

In [ ]:
winner     = race_results.iloc[0]['FullName']
p2         = race_results.iloc[1]['FullName']
p3         = race_results.iloc[2]['FullName']
event_name = race.event['EventName']
pole       = quali_df.iloc[0]['FullName']

post = f"""
🏁 {event_name} {YEAR} — Full Weekend in Data

[HOOK — e.g.: {pole} took pole, but was it really their weekend?]

🔵 Qualifying:
· Pole: {pole} | [delta between P1 and P2 from chart]
· [Key sector time insight]

{'🟡 Sprint: [key sprint insight]' if HAS_SPRINT else ''}

🔴 Race:
· P1 {winner} · P2 {p2} · P3 {p3}
· [Tyre strategy insight]
· [Telemetry insight: {DRIVER_1} vs {DRIVER_2}]

📊 Biggest mover: [from positions gained chart]

💡 [1-2 sentence story of the weekend in data]

What was your highlight of the weekend? 👇

#Formula1 #F1 #DataAnalysis #DataScience #Motorsport #FastF1 #Python #{event_name.replace(' ', '')}
"""

print(post)

---
## Assets generated

| File | Section | LinkedIn use |
|------|---------|-------------|
| `Q1_qualifying_times.png` | Qualifying | Image 1 |
| `Q2_sector_times.png` | Qualifying | Image 2 |
| `SQ1_shootout_times.png` | Sprint Shootout | Image 3 (Sprint only) |
| `S1_sprint_results.png` | Sprint Race | Image 4 (Sprint only) |
| `01_race_results.png` | Race | Image 3 or 5 |
| `02_lap_time_evolution.png` | Race | Image 4 or 6 |
| `03_tyre_strategy.png` | Race | High engagement |
| `04_telemetry_*.png` | Race | Most technical |
| `W1_positions_gained.png` | Weekend | Strong visual |
| `W2_pace_comparison.png` | Weekend | Clean summary |

**Commit:**
```bash
git add 2026/R{round}_{location}/
git commit -m "feat: {event_name} {year} weekend analysis"
git push
```